
# Milestone 06 — Model Improvement & Class Imbalance Handling

## Project: Bank Account Fraud Detection — Deep Learning & MLOps

### Objectives

Improve our Milestone 05 PyTorch MLP through controlled experiments while maintaining reproducibility and preventing data leakage.

### Starting Benchmarks

| Model | Validation PR-AUC |
|---|---:|
| DummyClassifier | 0.013405 |
| Logistic Regression | 0.154819 |
| PyTorch MLP — Epoch 5 | 0.157469 |
| HistGradientBoosting | 0.166961 |

**Primary evaluation metric:** PR-AUC (Average Precision)

### Experiment Phases

- **06A:** Experiment tracking and configuration management
- **06B:** Google Colab integration and CPU/GPU benchmarking
- **06C:** Class imbalance strategies and hyperparameter tuning
- **06D:** Model selection, reproducibility checks, and reporting

### Data Leakage Rules

- Training: Months 0–5
- Validation: Month 6
- Test: Month 7 — SEALED
- Fit preprocessing on training data only.
- Do not access the test set during model development.
- Apply any sampling or class-balancing procedures to training data only.

### Experiment Standards

Each experiment must record:

- Experiment identifier and random seed
- Model architecture and hyperparameters
- Loss function, optimizer, and learning rate
- Class imbalance strategy
- Training and validation metrics
- Training device and execution duration
- Model selection decision and justification

### Success Criteria

1. Establish a reliable experiment-tracking workflow.
2. Compare training performance on local CPU and Google Colab GPU.
3. Evaluate multiple controlled model improvements.
4. Compare results against the fixed Milestone 05 baseline.
5. Preserve reproducibility and maintain the sealed test set.


### Environment & Reproducibility Setup

In [ ]:

# ============================================================
# Milestone 06 — Cell 2: Environment & Reproducibility
# ============================================================

from pathlib import Path
import sys
import random
import platform

import numpy as np
import pandas as pd
import torch
import sklearn

# 1. Locate our project directory
PROJECT_ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src/models/mlp.py").is_file()
        and (path / "notebooks/05_pytorch_mlp.ipynb").is_file()
    ),
    None
)

assert PROJECT_ROOT is not None, "Project root not found"

# 2. Make our reusable modules importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. Configure reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 4. Detect available hardware
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 5. Display environment information
print("PROJECT ENVIRONMENT")
print("=" * 55)

print("Project root:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nREPRODUCIBILITY")
print("=" * 55)

print("Random seed:", SEED)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nEnvironment verification: PASS")


### Experiment Configuration

In [ ]:

# ============================================================
# Milestone 06 — Cell 3: Experiment Configuration
# ============================================================

from dataclasses import dataclass, asdict
import json

@dataclass(frozen=True)
class ExperimentConfig:
    experiment_id: str = "m06_reference_mlp_config"

    # Reproducibility
    seed: int = 42
    device: str = str(DEVICE)

    # Model architecture
    input_features: int = 55
    hidden_layers: tuple[int, int] = (64, 32)
    dropout_rate: float = 0.10

    # Training hyperparameters
    epochs: int = 5
    batch_size: int = 1024
    learning_rate: float = 0.001
    optimizer: str = "Adam"
    loss_function: str = "BCEWithLogitsLoss"

    # Class imbalance strategy
    imbalance_strategy: str = "none"

    # Temporal data boundaries
    train_months: tuple[int, ...] = (0, 1, 2, 3, 4, 5)
    validation_months: tuple[int, ...] = (6,)
    test_month: int = 7
    test_status: str = "SEALED"


# Create the reference configuration
config = ExperimentConfig()

# Validate configuration
assert config.input_features == 55
assert config.epochs > 0
assert config.batch_size > 0
assert config.learning_rate > 0
assert 0 <= config.dropout_rate < 1

# Prevent accidental overlap between dataset splits
assert set(config.train_months).isdisjoint(
    config.validation_months
)
assert config.test_month not in (
    *config.train_months,
    *config.validation_months,
)
assert config.test_status == "SEALED"

# Display configuration as readable JSON
print("EXPERIMENT CONFIGURATION")
print("=" * 55)
print(json.dumps(asdict(config), indent=2))

print("\nConfiguration validation: PASS")
print("Temporal split protection: PASS")
print("Experiment tracking setup: PASS")


### Build Our Experiment Registry

In [ ]:

# ============================================================
# Milestone 06 — Cell 4: Experiment Registry
# ============================================================

# 1. Locate our verified Milestone 05 metadata
BASELINE_METADATA_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "milestone_05_reference"
    / "milestone_05_mlp_metadata.json"
)

assert BASELINE_METADATA_PATH.is_file(), (
    f"Baseline metadata not found: {BASELINE_METADATA_PATH}"
)

# 2. Load historical experiment information
with open(BASELINE_METADATA_PATH, "r", encoding="utf-8") as file:
    baseline_metadata = json.load(file)

architecture = baseline_metadata["architecture"]
training = baseline_metadata["training"]
results = baseline_metadata["validation_results"]

# 3. Confirm compatibility with our reference configuration
assert architecture["input_features"] == config.input_features
assert tuple(architecture["hidden_layers"]) == config.hidden_layers
assert architecture["dropout"] == config.dropout_rate
assert training["epochs"] == config.epochs
assert training["random_seed"] == config.seed
assert baseline_metadata["data"]["test_status"] == "SEALED"

# 4. Create our first registry record
baseline_record = {
    "experiment_id": "m05_fixed_budget_baseline",
    "source": "milestone_05_reference",
    "status": "completed_imported",
    "device": training["device"],
    "seed": training["random_seed"],
    "epochs": training["epochs"],
    "learning_rate": training["learning_rate"],
    "dropout_rate": architecture["dropout"],
    "imbalance_strategy": "none",
    "validation_pr_auc": float(results["pr_auc"]),
    "validation_roc_auc": float(results["roc_auc"]),
    "validation_loss": float(results["validation_loss"]),
}

# 5. Initialize registry
experiment_registry = [baseline_record]

# Convert to a DataFrame for convenient comparisons
registry_df = pd.DataFrame(experiment_registry)

# 6. Verify the registry
assert len(registry_df) == 1
assert registry_df["experiment_id"].is_unique
assert registry_df["validation_pr_auc"].between(0, 1).all()
assert registry_df["status"].iloc[0] == "completed_imported"

# 7. Display the initial registry
print("MILESTONE 06 — EXPERIMENT REGISTRY")
print("=" * 65)

display(
    registry_df[
        [
            "experiment_id",
            "device",
            "imbalance_strategy",
            "epochs",
            "validation_pr_auc",
            "validation_roc_auc",
            "status",
        ]
    ].round(6)
)

print("\nHistorical baseline imported: PASS")
print("Experiment registry initialized: PASS")
print("Month-7 test set remains: SEALED")


### Build a Safe Experiment Registration Function

In [ ]:

# ============================================================
# Milestone 06 — Cell 5: Safe Experiment Registration
# ============================================================

REQUIRED_EXPERIMENT_FIELDS = {
    "experiment_id",
    "source",
    "status",
    "device",
    "seed",
    "epochs",
    "learning_rate",
    "dropout_rate",
    "imbalance_strategy",
    "validation_pr_auc",
    "validation_roc_auc",
    "validation_loss",
}


def register_experiment(registry: list, record: dict) -> None:
    """Validate and register one completed experiment."""

    # 1. Require all expected fields
    missing = REQUIRED_EXPERIMENT_FIELDS - record.keys()

    if missing:
        raise ValueError(
            f"Missing experiment fields: {sorted(missing)}"
        )

    # 2. Prevent accidental test metric registration
    if any(str(key).startswith("test_") for key in record):
        raise ValueError(
            "Test-set metrics are prohibited during Milestone 06."
        )

    # 3. Require unique experiment identifiers
    existing_ids = {r["experiment_id"] for r in registry}

    if record["experiment_id"] in existing_ids:
        raise ValueError(
            f"Duplicate experiment ID: {record['experiment_id']}"
        )

    # 4. Validate numerical metrics
    for metric in (
        "validation_pr_auc",
        "validation_roc_auc",
        "validation_loss",
    ):
        value = float(record[metric])

        if not np.isfinite(value):
            raise ValueError(f"{metric} must be finite")

        if metric in (
            "validation_pr_auc",
            "validation_roc_auc",
        ):
            if not 0.0 <= value <= 1.0:
                raise ValueError(f"{metric} must be between 0 and 1")

        if metric == "validation_loss" and value < 0:
            raise ValueError("Validation loss cannot be negative")

    # 5. Register only after all checks pass
    registry.append(record.copy())


# ------------------------------------------------------------
# Verify the registry safeguards without adding a fake run
# ------------------------------------------------------------

initial_count = len(experiment_registry)

# Test duplicate prevention using our imported baseline
try:
    register_experiment(experiment_registry, baseline_record)

except ValueError as error:
    assert "Duplicate experiment ID" in str(error)
    print("Duplicate experiment protection: PASS")

else:
    raise AssertionError("Duplicate experiment was accepted")


# Confirm the registry remains unchanged
assert len(experiment_registry) == initial_count == 1

# Confirm no test metrics appear in the registry
assert all(
    not any(str(key).startswith("test_") for key in record)
    for record in experiment_registry
)

print("Registry integrity: PASS")
print("Test metric exclusion: PASS")
print("Safe experiment registration: PASS")


### Experiment Comparison & Portable Results

In [ ]:

# ============================================================
# Milestone 06 — Cell 6: Experiment Comparison & Portable Results
# ============================================================

# 1. Retrieve verified historical benchmarks
BASELINE_EXPERIMENT_ID = "m05_fixed_budget_baseline"

baseline_pr_auc = float(
    baseline_record["validation_pr_auc"]
)

hgb_pr_auc = float(
    results["histgradientboosting_pr_auc"]
)

# 2. Rebuild the table from our current registry
comparison_df = pd.DataFrame(experiment_registry).copy()

assert comparison_df["experiment_id"].is_unique
assert BASELINE_EXPERIMENT_ID in comparison_df["experiment_id"].values

# 3. Calculate improvement over the MLP baseline
comparison_df["delta_vs_m05"] = (
    comparison_df["validation_pr_auc"] - baseline_pr_auc
)

# 4. Calculate how far each model is from the HGB benchmark
comparison_df["gap_to_hgb"] = (
    hgb_pr_auc - comparison_df["validation_pr_auc"]
)

# 5. Identify experiments that outperform HGB
comparison_df["beats_hgb"] = (
    comparison_df["validation_pr_auc"] > hgb_pr_auc
)

# 6. Sort experiments by our primary evaluation metric
comparison_df = comparison_df.sort_values(
    by="validation_pr_auc",
    ascending=False
).reset_index(drop=True)

# 7. Create a portable experiment snapshot
portable_registry = {
    "schema_version": 1,
    "milestone": "06",
    "primary_metric": "validation_pr_auc",
    "baseline_experiment_id": BASELINE_EXPERIMENT_ID,
    "baseline_pr_auc": baseline_pr_auc,
    "hgb_benchmark_pr_auc": hgb_pr_auc,
    "holdout_status": "SEALED",
    "reference_config": asdict(config),
    "experiments": [record.copy() for record in experiment_registry],
}

# Ensure the entire snapshot can be serialized as JSON
portable_registry_json = json.dumps(
    portable_registry,
    indent=2,
    allow_nan=False
)

# 8. Verify calculated comparisons
baseline_row = comparison_df.loc[
    comparison_df["experiment_id"] == BASELINE_EXPERIMENT_ID
].iloc[0]

assert np.isclose(baseline_row["delta_vs_m05"], 0.0)
assert baseline_row["gap_to_hgb"] > 0
assert portable_registry["holdout_status"] == "SEALED"
assert len(portable_registry["experiments"]) == 1

# 9. Display the results
print("MILESTONE 06 — EXPERIMENT COMPARISON")
print("=" * 65)

display(
    comparison_df[
        [
            "experiment_id",
            "validation_pr_auc",
            "delta_vs_m05",
            "gap_to_hgb",
            "beats_hgb",
        ]
    ].round(6)
)

print("\nEXPERIMENT TRACKING")
print("Registered experiments:", len(experiment_registry))
print("MLP reference PR-AUC:", round(baseline_pr_auc, 6))
print("HGB benchmark PR-AUC:", round(hgb_pr_auc, 6))

print("\nComparison calculations: PASS")
print("Portable JSON serialization: PASS")
print("Holdout status: SEALED")
print("Experiment comparison setup: PASS")


### Safely Export the Experiment Registry

In [ ]:

# ============================================================
# Milestone 06 — Cell 7: Safe Experiment Registry Export
# ============================================================

import hashlib

# 1. Define the destination
EXPORT_DIR = PROJECT_ROOT / "artifacts" / "milestone_06"

REGISTRY_EXPORT_PATH = (
    EXPORT_DIR / "experiment_registry_v01.json"
)

# 2. Verify our in-memory experiment registry
assert len(experiment_registry) == 1
assert portable_registry["holdout_status"] == "SEALED"
assert portable_registry["primary_metric"] == "validation_pr_auc"

assert all(
    not any(str(key).startswith("test_") for key in record)
    for record in portable_registry["experiments"]
)

# 3. Prevent accidental overwriting
if REGISTRY_EXPORT_PATH.exists():
    raise FileExistsError(
        f"Experiment registry already exists: {REGISTRY_EXPORT_PATH}. "
        "Use a new versioned filename instead of overwriting."
    )

# 4. Create the output directory
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 5. Save JSON with exclusive file creation
export_text = portable_registry_json + "\n"

with REGISTRY_EXPORT_PATH.open(
    "x", encoding="utf-8", newline="\n"
) as file:
    file.write(export_text)

# 6. Reload the exported JSON
with REGISTRY_EXPORT_PATH.open(
    "r", encoding="utf-8"
) as file:
    reloaded_registry = json.load(file)

# 7. Validate saved content against the original snapshot
expected_registry = json.loads(portable_registry_json)

assert reloaded_registry == expected_registry
assert reloaded_registry["holdout_status"] == "SEALED"
assert len(reloaded_registry["experiments"]) == 1

# 8. Verify byte-level integrity
saved_bytes = REGISTRY_EXPORT_PATH.read_bytes()
expected_bytes = export_text.encode("utf-8")

assert saved_bytes == expected_bytes

registry_sha256 = hashlib.sha256(saved_bytes).hexdigest()

# 9. Display verification results
print("MILESTONE 06 — EXPERIMENT REGISTRY EXPORT")
print("=" * 60)

print("File:", REGISTRY_EXPORT_PATH.name)
print("Location:", REGISTRY_EXPORT_PATH)
print("File exists:", REGISTRY_EXPORT_PATH.is_file())
print("Registered experiments:", len(reloaded_registry["experiments"]))
print("File size:", len(saved_bytes), "bytes")
print("SHA-256:", registry_sha256)

print("\nRegistry JSON reload: PASS")
print("File integrity verification: PASS")
print("Overwrite protection: ENABLED")
print("Holdout status: SEALED")
print("Experiment registry export: PASS")


### Reconstruct Our 55-Feature Development Arrays

In [ ]:

# ============================================================
# Milestone 06 — Cell 8: Prepare Colab Development Arrays
# ============================================================

import joblib
import json

from src.data import splitting
from src.features import preprocessing

# 1. Reuse existing project configurations
raw_path = PROJECT_ROOT / "data/raw/Base.csv"
split_path = PROJECT_ROOT / "configs/split_config.json"
prep_path = PROJECT_ROOT / "configs/preprocessing_config.json"

# Trusted, locally generated Milestone 05 artifacts
preprocessor_path = (
    PROJECT_ROOT / "models/milestone_05_preprocessor.joblib"
)

metadata_path = (
    PROJECT_ROOT
    / "artifacts/milestone_05_reference"
    / "milestone_05_mlp_metadata.json"
)

for path in (
    raw_path,
    split_path,
    prep_path,
    preprocessor_path,
    metadata_path,
):
    assert path.is_file(), f"Missing file: {path}"

split_cfg = splitting.load_split_config(split_path)
prep_cfg = preprocessing.load_preprocessing_config(prep_path)

# 2. Verify temporal boundaries
assert split_cfg["train"]["months"] == list(range(6))
assert split_cfg["validation"]["months"] == [6]
assert split_cfg["test"]["months"] == [7]
assert split_cfg["test"]["sealed"] is True

# 3. Create training and validation populations
raw = pd.read_csv(raw_path)

labels = splitting.assign_split(
    raw, config=split_cfg
)

train_df = raw.loc[labels.eq("train")].copy()
val_df = raw.loc[labels.eq("validation")].copy()

# Immediately release raw data and split labels
del raw, labels

target = prep_cfg["target_column"]
month_field = split_cfg["temporal_field"]

# 4. Verify populations and fraud counts
assert len(train_df) == split_cfg["train"]["records"]
assert len(val_df) == split_cfg["validation"]["records"]

assert int(train_df[target].sum()) == (
    split_cfg["train"]["fraud_cases"]
)
assert int(val_df[target].sum()) == (
    split_cfg["validation"]["fraud_cases"]
)

assert set(train_df[month_field].unique()) == set(range(6))
assert set(val_df[month_field].unique()) == {6}

# 5. Prepare the original 34 model features
train_features = preprocessing.prepare_model_features(
    train_df, prep_cfg
)
val_features = preprocessing.prepare_model_features(
    val_df, prep_cfg
)

m06_y_train = train_df[target].to_numpy(
    dtype=np.float32
)
m06_y_validation = val_df[target].to_numpy(
    dtype=np.float32
)

assert train_features.shape[1] == 34
assert val_features.shape[1] == 34
assert train_features.columns.equals(val_features.columns)

assert target not in train_features.columns
assert month_field not in train_features.columns

# Raw split DataFrames are no longer needed
del train_df, val_df

# 6. Load the trusted fitted preprocessor
# DO NOT fit or refit it.
fitted_preprocessor = joblib.load(preprocessor_path)

m06_X_train = np.asarray(
    fitted_preprocessor.transform(train_features),
    dtype=np.float32
)

m06_X_validation = np.asarray(
    fitted_preprocessor.transform(val_features),
    dtype=np.float32
)

m06_feature_names = list(
    fitted_preprocessor.get_feature_names_out()
)

# 7. Verify against Milestone 05 metadata
with open(metadata_path, "r", encoding="utf-8") as file:
    reference_metadata = json.load(file)

assert (
    m06_feature_names
    == reference_metadata["data"]["processed_feature_names"]
)

assert m06_X_train.shape == (794989, 55)
assert m06_X_validation.shape == (108168, 55)

assert m06_y_train.shape == (794989,)
assert m06_y_validation.shape == (108168,)

assert m06_X_train.dtype == np.float32
assert m06_X_validation.dtype == np.float32

assert np.isfinite(m06_X_train).all()
assert np.isfinite(m06_X_validation).all()

assert set(np.unique(m06_y_train)) == {0.0, 1.0}
assert set(np.unique(m06_y_validation)) == {0.0, 1.0}

# 8. Display final verification
print("MILESTONE 06 — PORTABLE DATA PREPARATION")
print("=" * 60)

print("Training features:", m06_X_train.shape)
print("Validation features:", m06_X_validation.shape)
print("Training labels:", m06_y_train.shape)
print("Validation labels:", m06_y_validation.shape)

print("\nTraining fraud cases:", int(m06_y_train.sum()))
print("Validation fraud cases:", int(m06_y_validation.sum()))
print("Feature dtype:", m06_X_train.dtype)
print("Feature count:", len(m06_feature_names))

print("\nFitted preprocessor reused: PASS")
print("Feature order matches Milestone 05: PASS")
print("Training/validation alignment: PASS")
print("Non-finite feature check: PASS")
print("Month-7 test data excluded: PASS")
print("Portable data preparation: PASS")


### Export Portable NumPy Arrays
**What we're doing**

We'll create one compressed .npz archive containing:

**Array	Contents**
- X_train	794,989 × 55 training features
- y_train	794,989 training labels
- X_validation	108,168 × 55 validation features
- y_validation	108,168 validation labels
- feature_names	Exact order of all 55 inputs

In [ ]:

# ============================================================
# Milestone 06 — Cell 9: Export Portable Colab Arrays
# ============================================================

import hashlib
import subprocess

# 1. Define versioned archive path
COLAB_EXPORT_DIR = PROJECT_ROOT / "artifacts" / "milestone_06"

COLAB_ARRAYS_PATH = (
    COLAB_EXPORT_DIR / "colab_dev_arrays_v01.npz"
)

# 2. Verify the file is excluded from Git
relative_path = COLAB_ARRAYS_PATH.relative_to(
    PROJECT_ROOT
).as_posix()

ignore_check = subprocess.run(
    ["git", "check-ignore", "--quiet", relative_path],
    cwd=PROJECT_ROOT,
    check=False,
)

assert ignore_check.returncode == 0, (
    "STOP: Export destination is not confirmed "
    "as Git-ignored."
)

# 3. Validate the arrays before export
assert m06_X_train.shape == (794989, 55)
assert m06_X_validation.shape == (108168, 55)
assert m06_y_train.shape == (794989,)
assert m06_y_validation.shape == (108168,)

assert m06_X_train.dtype == np.float32
assert m06_X_validation.dtype == np.float32

assert len(m06_feature_names) == 55
assert int(m06_y_train.sum()) == 8151
assert int(m06_y_validation.sum()) == 1450

assert split_cfg["test"]["sealed"] is True

# 4. Prevent accidental overwriting
if COLAB_ARRAYS_PATH.exists():
    raise FileExistsError(
        f"Export already exists: {COLAB_ARRAYS_PATH}. "
        "Choose a new version instead."
    )

COLAB_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 5. Export only training and validation data
with COLAB_ARRAYS_PATH.open("xb") as archive:
    np.savez_compressed(
        archive,
        X_train=m06_X_train,
        y_train=m06_y_train,
        X_validation=m06_X_validation,
        y_validation=m06_y_validation,
        feature_names=np.asarray(m06_feature_names),
    )

# 6. Reload and verify the archive
with np.load(
    COLAB_ARRAYS_PATH,
    allow_pickle=False,
) as saved:

    expected_keys = {
        "X_train",
        "y_train",
        "X_validation",
        "y_validation",
        "feature_names",
    }

    assert set(saved.files) == expected_keys

    assert np.array_equal(
        saved["X_train"], m06_X_train
    )
    assert np.array_equal(
        saved["y_train"], m06_y_train
    )
    assert np.array_equal(
        saved["X_validation"], m06_X_validation
    )
    assert np.array_equal(
        saved["y_validation"], m06_y_validation
    )
    assert saved["feature_names"].tolist() == m06_feature_names

# 7. Calculate SHA-256 without loading file into memory
sha256 = hashlib.sha256()

with COLAB_ARRAYS_PATH.open("rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b"",
    ):
        sha256.update(chunk)

COLAB_ARRAYS_SHA256 = sha256.hexdigest()

# 8. Display export results
print("MILESTONE 06 — COLAB ARRAY EXPORT")
print("=" * 60)

print("Archive:", COLAB_ARRAYS_PATH.name)
print(
    "File size (MiB):",
    round(COLAB_ARRAYS_PATH.stat().st_size / (1024 ** 2), 2)
)
print("SHA-256:", COLAB_ARRAYS_SHA256)

print("\nGit exclusion: PASS")
print("All five arrays verified: PASS")
print("Array contents match source: PASS")
print("Overwrite protection: ENABLED")
print("Month-7 data included: NO")
print("Portable array export: PASS")


### Create the Data Integrity Manifest

In [ ]:

# ============================================================
# Milestone 06 — Cell 10: Portable Data Integrity Manifest
# ============================================================

import platform
import sklearn

# 1. Define the manifest destination
MANIFEST_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "milestone_06"
    / "colab_dev_manifest_v01.json"
)

# 2. Verify the exported archive still exists
assert COLAB_ARRAYS_PATH.is_file()

# Recalculate SHA-256 from the actual file
archive_hash = hashlib.sha256()

with COLAB_ARRAYS_PATH.open("rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b"",
    ):
        archive_hash.update(chunk)

assert archive_hash.hexdigest() == COLAB_ARRAYS_SHA256

# 3. Record provenance of the fitted preprocessor
preprocessor_hash = hashlib.sha256(
    preprocessor_path.read_bytes()
).hexdigest()

# 4. Build the portable manifest
data_manifest = {
    "schema_version": 1,
    "milestone": "06",
    "archive_name": COLAB_ARRAYS_PATH.name,
    "archive_sha256": COLAB_ARRAYS_SHA256,

    "source": {
        "dataset": "Base.csv",
        "preprocessing": "Milestone 05 fitted preprocessor",
        "preprocessor_sha256": preprocessor_hash,
        "python_version": platform.python_version(),
        "scikit_learn_version": sklearn.__version__,
    },

    "training": {
        "months": split_cfg["train"]["months"],
        "feature_shape": list(m06_X_train.shape),
        "target_shape": list(m06_y_train.shape),
        "fraud_cases": int(m06_y_train.sum()),
    },

    "validation": {
        "months": split_cfg["validation"]["months"],
        "feature_shape": list(m06_X_validation.shape),
        "target_shape": list(m06_y_validation.shape),
        "fraud_cases": int(m06_y_validation.sum()),
    },

    "features": {
        "count": len(m06_feature_names),
        "dtype": str(m06_X_train.dtype),
        "ordered_names": [
            str(name) for name in m06_feature_names
        ],
    },

    "holdout": {
        "month": 7,
        "status": "SEALED",
        "included_in_archive": False,
    },
}

# 5. Validate manifest before saving
assert data_manifest["training"]["feature_shape"] == [794989, 55]
assert data_manifest["validation"]["feature_shape"] == [108168, 55]
assert data_manifest["features"]["count"] == 55
assert data_manifest["training"]["fraud_cases"] == 8151
assert data_manifest["validation"]["fraud_cases"] == 1450
assert data_manifest["holdout"]["included_in_archive"] is False

assert data_manifest["features"]["ordered_names"] == (
    reference_metadata["data"]["processed_feature_names"]
)

# 6. Refuse to overwrite existing metadata
if MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Manifest already exists: {MANIFEST_PATH}. "
        "Use a new versioned filename."
    )

# 7. Save and reload the manifest
with MANIFEST_PATH.open("x", encoding="utf-8") as file:
    json.dump(data_manifest, file, indent=2)
    file.write("\n")

with MANIFEST_PATH.open("r", encoding="utf-8") as file:
    loaded_manifest = json.load(file)

assert loaded_manifest == data_manifest

# 8. Display verification
print("MILESTONE 06 — DATA INTEGRITY MANIFEST")
print("=" * 60)

print("Manifest:", MANIFEST_PATH.name)
print("Archive:", loaded_manifest["archive_name"])
print("Archive SHA-256:", loaded_manifest["archive_sha256"])
print("Feature count:", loaded_manifest["features"]["count"])
print("Training records:", loaded_manifest["training"]["feature_shape"][0])
print("Validation records:", loaded_manifest["validation"]["feature_shape"][0])

print("\nArchive checksum verified: PASS")
print("Feature ordering verified: PASS")
print("Preprocessing provenance recorded: PASS")
print("Manifest save/reload: PASS")
print("Test split: SEALED")
print("Data integrity manifest: PASS")


### Register Both GPU Experiments

In [ ]:

# ============================================================
# Milestone 06 — Register Verified Colab GPU Experiments
# ============================================================

import json
import hashlib
import zipfile
import pandas as pd

# 1. Confirm our original registry is available
assert "experiment_registry" in globals(), (
    "Run the existing experiment registry setup cells first."
)
assert "register_experiment" in globals(), (
    "Run the safe experiment registration cell first."
)

assert len(experiment_registry) == 1
assert experiment_registry[0]["experiment_id"] == (
    "m05_fixed_budget_baseline"
)

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "milestone_06"

# 2. Define the two verified exports
GPU_EXPORTS = [
    {
        "zip": "m06_unweighted_gpu_reference_v01.zip",
        "history": "m06_unweighted_gpu_history_v01.json",
        "manifest": "m06_unweighted_gpu_manifest_v01.json",
        "sha256": "352bdd96315364a7e75d1a0536937343b33d46029f7122dbc8e81927d419b447",
        "imbalance": "none",
        "loss_key": "val_loss",
        "expected_pr_auc": 0.160393,
    },
    {
        "zip": "m06_class_weighted_gpu_v01.zip",
        "history": "m06_class_weighted_history_v01.json",
        "manifest": "m06_class_weighted_manifest_v01.json",
        "sha256": "a426d3e7f07405172d45113a2c3c9b717e5366d3c54ba27259bd894a2ee64b55",
        "imbalance": "positive_class_weight",
        "loss_key": "val_unweighted_loss",
        "expected_pr_auc": 0.163862,
    },
]

# 3. Read and verify each ZIP before registering
def load_gpu_experiment(spec):
    path = ARTIFACT_DIR / spec["zip"]
    assert path.is_file(), f"Missing archive: {path}"

    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    assert digest.hexdigest() == spec["sha256"]

    with zipfile.ZipFile(path) as archive:
        assert archive.testzip() is None

        history = json.loads(
            archive.read(spec["history"])
        )
        manifest = json.loads(
            archive.read(spec["manifest"])
        )

        for filename, expected_hash in manifest["files"].items():
            actual_hash = hashlib.sha256(
                archive.read(filename)
            ).hexdigest()
            assert actual_hash == expected_hash

    # Verify experiment provenance
    assert history["source_commit"] == "cf443a5"
    assert history["test_status"] == "SEALED"
    assert history["train_months"] == [0, 1, 2, 3, 4, 5]
    assert history["validation_months"] == [6]
    assert history["feature_count"] == 55
    assert history["parameter_count"] == 5697
    assert history["epochs"] == 5
    assert history["seed"] == 42
    assert len(history["epoch_history"]) == 5

    final = history["final_validation"]

    assert final["epoch"] == 5
    assert abs(
        final["val_pr_auc"] - spec["expected_pr_auc"]
    ) < 0.000001

    # Both validation losses are UNWEIGHTED and comparable.
    record = {
        "experiment_id": history["experiment_id"],
        "source": "colab_verified_archive",
        "status": "completed_verified",
        "device": history["device"],
        "seed": history["seed"],
        "epochs": history["epochs"],
        "learning_rate": history["learning_rate"],
        "dropout_rate": 0.10,
        "imbalance_strategy": spec["imbalance"],
        "validation_pr_auc": float(final["val_pr_auc"]),
        "validation_roc_auc": float(final["val_roc_auc"]),
        "validation_loss": float(final[spec["loss_key"]]),
        "source_commit": history["source_commit"],
        "artifact_path": str(path.relative_to(PROJECT_ROOT)),
        "artifact_sha256": spec["sha256"],
    }

    return record

# 4. Validate both records in a temporary registry
# The original registry is unchanged if either record fails.
proposed_registry = [
    record.copy() for record in experiment_registry
]

for specification in GPU_EXPORTS:
    record = load_gpu_experiment(specification)
    register_experiment(proposed_registry, record)

# 5. Commit both verified records together
assert len(proposed_registry) == 3

experiment_registry = proposed_registry
registry_df = pd.DataFrame(experiment_registry)

assert registry_df["experiment_id"].is_unique
assert registry_df["validation_pr_auc"].between(0, 1).all()

# 6. Display updated registry
print("MILESTONE 06 — EXPERIMENT REGISTRY")
print("=" * 65)

display(
    registry_df[
        [
            "experiment_id",
            "device",
            "imbalance_strategy",
            "epochs",
            "validation_pr_auc",
            "validation_roc_auc",
            "status",
        ]
    ].round(6)
)

print("\nHistorical M05 baseline preserved: PASS")
print("Unweighted GPU experiment registered: PASS")
print("Class-weighted GPU experiment registered: PASS")
print("Archive integrity verification: PASS")
print("Duplicate experiment protection: PASS")
print("Month-7 test set: SEALED")
print("Step 06.30B: PASS")


### Export Experiment Registry v02

In [ ]:

# ============================================================
# Milestone 06 — Cell: Export Experiment Registry v02
# ============================================================

import json
import hashlib
from pathlib import Path

# 1. Verify the registered experiments
expected_ids = {
    "m05_fixed_budget_baseline",
    "m06_gpu_reference_unweighted_v01",
    "m06_class_weighted_gpu_v01",
}

actual_ids = {
    record["experiment_id"]
    for record in experiment_registry
}

assert len(experiment_registry) == 3
assert actual_ids == expected_ids

# Prevent test metrics from entering the registry
assert all(
    not any(str(key).startswith("test_") for key in record)
    for record in experiment_registry
)

# 2. Prepare a portable registry document
registry_document = {
    "registry_version": 2,
    "project": "bank-account-fraud-dl-mlops",
    "milestone": "06_model_improvement",
    "source_notebook": "notebooks/06_model_improvement.ipynb",
    "primary_metric": "validation_pr_auc",
    "selection_epoch": 5,
    "train_months": [0, 1, 2, 3, 4, 5],
    "validation_months": [6],
    "holdout_status": "SEALED",
    "experiment_count": len(experiment_registry),
    "experiments": [
        record.copy()
        for record in experiment_registry
    ],
}

# 3. Define the versioned output file
export_path = (
    PROJECT_ROOT
    / "artifacts"
    / "milestone_06"
    / "experiment_registry_v02.json"
)

assert export_path.parent.is_dir(), (
    "Milestone 06 artifact directory is missing."
)

# Never overwrite an existing registry snapshot
assert not export_path.exists(), (
    f"Registry v02 already exists: {export_path}"
)

# 4. Export safely
with export_path.open("x", encoding="utf-8") as file:
    json.dump(
        registry_document,
        file,
        indent=2,
        allow_nan=False,
    )

# 5. Independently reload the saved file
with export_path.open("r", encoding="utf-8") as file:
    restored_registry = json.load(file)

assert restored_registry == registry_document

assert restored_registry["experiment_count"] == 3
assert restored_registry["holdout_status"] == "SEALED"

# 6. Generate SHA-256 checksum
sha256 = hashlib.sha256()

with export_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

# 7. Display results
print("MILESTONE 06 — REGISTRY V02 EXPORT")
print("=" * 60)

print("File:", export_path.name)
print("Registered experiments:", len(experiment_registry))

print("\nVALIDATION PR-AUC")

for record in experiment_registry:
    print(
        f"{record['experiment_id']}: "
        f"{record['validation_pr_auc']:.6f}"
    )

print("\nSHA-256:", sha256.hexdigest())

print("\nRegistry v02 created: PASS")
print("JSON reload verification: PASS")
print("Three experiment records: PASS")
print("Original registry v01: PRESERVED")
print("Month-7 test set: SEALED")
print("Step 06.30C: PASS")


### Register the Weight-25 GPU Experiment

In [ ]:

# ============================================================
# Milestone 06 — Step 06.33E: Register Weight-25 Experiment
# ============================================================

import json
import hashlib
import zipfile
import pandas as pd

# 1. Verify the existing registry is available
assert "experiment_registry" in globals(), (
    "Experiment registry not loaded. Stop and restore "
    "the existing registry before continuing."
)

assert "register_experiment" in globals(), (
    "Registration function not loaded."
)

expected_existing_ids = {
    "m05_fixed_budget_baseline",
    "m06_gpu_reference_unweighted_v01",
    "m06_class_weighted_gpu_v01",
}

assert len(experiment_registry) == 3
assert {
    record["experiment_id"]
    for record in experiment_registry
} == expected_existing_ids

# 2. Locate the verified Weight-25 artifact
archive_path = (
    PROJECT_ROOT
    / "artifacts"
    / "milestone_06"
    / "m06_weight25_gpu_v01.zip"
)

assert archive_path.is_file()

expected_zip_sha256 = (
    "644255d89b7d47eb52ae9418e7af0b22"
    "f65817271be703869123f8a3c0547814"
)

digest = hashlib.sha256()

with archive_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        digest.update(chunk)

assert digest.hexdigest() == expected_zip_sha256

# 3. Read archived history and verify internal checksums
with zipfile.ZipFile(archive_path, "r") as archive:

    assert archive.testzip() is None

    assert set(archive.namelist()) == {
        "m06_weight25_gpu_v01.pt",
        "m06_weight25_history_v01.json",
        "m06_weight25_manifest_v01.json",
    }

    history = json.loads(
        archive.read("m06_weight25_history_v01.json")
    )

    manifest = json.loads(
        archive.read("m06_weight25_manifest_v01.json")
    )

    for filename, expected_hash in manifest["files"].items():

        actual_hash = hashlib.sha256(
            archive.read(filename)
        ).hexdigest()

        assert actual_hash == expected_hash

# 4. Verify experiment provenance and configuration
assert history["experiment_id"] == "m06_weight25_gpu_v01"
assert history["source_commit"] == "cf443a5"

assert history["architecture"] == "FraudMLP"
assert history["feature_count"] == 55
assert history["parameter_count"] == 5697

assert history["seed"] == 42
assert history["epochs"] == 5
assert history["batch_size"] == 1024
assert history["learning_rate"] == 0.001
assert history["positive_class_weight"] == 25.0

assert history["train_months"] == [0, 1, 2, 3, 4, 5]
assert history["validation_months"] == [6]
assert history["test_status"] == "SEALED"

assert len(history["epoch_history"]) == 5

# 5. Verify fixed-epoch validation metrics
final = history["final_validation"]

assert final["epoch"] == 5

assert abs(
    final["val_pr_auc"] - 0.161925
) < 0.000001

assert abs(
    final["val_roc_auc"] - 0.885421
) < 0.000001

assert abs(
    final["val_unweighted_loss"] - 0.163284
) < 0.000001

# 6. Construct the registry record
weight25_record = {
    "experiment_id": history["experiment_id"],
    "source": "colab_verified_archive",
    "status": "completed_verified",
    "device": history["device"],
    "seed": history["seed"],
    "epochs": history["epochs"],
    "learning_rate": history["learning_rate"],
    "dropout_rate": 0.10,
    "imbalance_strategy": "positive_class_weight_25",
    "validation_pr_auc": float(final["val_pr_auc"]),
    "validation_roc_auc": float(final["val_roc_auc"]),
    "validation_loss": float(final["val_unweighted_loss"]),
    "source_commit": history["source_commit"],
    "artifact_path": str(
        archive_path.relative_to(PROJECT_ROOT)
    ),
    "artifact_sha256": expected_zip_sha256,
}

# 7. Validate on a copy first
proposed_registry = [
    record.copy()
    for record in experiment_registry
]

register_experiment(
    proposed_registry,
    weight25_record,
)

assert len(proposed_registry) == 4

assert len({
    record["experiment_id"]
    for record in proposed_registry
}) == 4

# 8. Update the in-memory registry only after validation
experiment_registry = proposed_registry
registry_df = pd.DataFrame(experiment_registry)

print("MILESTONE 06 — FOUR-EXPERIMENT REGISTRY")
print("=" * 65)

display(
    registry_df[
        [
            "experiment_id",
            "imbalance_strategy",
            "epochs",
            "validation_pr_auc",
            "validation_roc_auc",
            "status",
        ]
    ].round(6)
)

print("\nWeight-25 archive integrity: PASS")
print("Weight-25 configuration: PASS")
print("Weight-25 experiment registered: PASS")
print("Previous three experiments preserved: PASS")
print("Registry experiment count:", len(experiment_registry))
print("Month-7 test set: SEALED")
print("Step 06.33E: PASS")


### Export Experiment Registry v03

In [ ]:

# ============================================================
# Milestone 06 — Step 06.33F: Export Registry v03
# ============================================================

import json
import hashlib
from pathlib import Path

# 1. Verify the four registered experiments
expected_ids = [
    "m05_fixed_budget_baseline",
    "m06_gpu_reference_unweighted_v01",
    "m06_class_weighted_gpu_v01",
    "m06_weight25_gpu_v01",
]

actual_ids = [
    record["experiment_id"]
    for record in experiment_registry
]

assert actual_ids == expected_ids
assert len(set(actual_ids)) == 4

# 2. Confirm the sealed test set is protected
assert all(
    not any(str(key).startswith("test_") for key in record)
    for record in experiment_registry
)

# 3. Locate existing versioned registries
artifact_dir = (
    PROJECT_ROOT / "artifacts" / "milestone_06"
)

v01_path = artifact_dir / "experiment_registry_v01.json"
v02_path = artifact_dir / "experiment_registry_v02.json"
v03_path = artifact_dir / "experiment_registry_v03.json"

assert v01_path.is_file()
assert v02_path.is_file()
assert not v03_path.exists(), (
    "Registry v03 already exists. Do not overwrite it."
)

# 4. Verify the original three records remain unchanged
with v02_path.open("r", encoding="utf-8") as file:
    previous_registry = json.load(file)

assert previous_registry["registry_version"] == 2
assert previous_registry["experiment_count"] == 3

assert (
    previous_registry["experiments"]
    == experiment_registry[:3]
), "Previous experiment records have changed."

# 5. Build the new registry snapshot
registry_v03 = {
    "registry_version": 3,
    "project": "bank-account-fraud-dl-mlops",
    "milestone": "06_model_improvement",
    "source_notebook": "notebooks/06_model_improvement.ipynb",
    "primary_metric": "validation_pr_auc",
    "selection_epoch": 5,
    "train_months": [0, 1, 2, 3, 4, 5],
    "validation_months": [6],
    "holdout_status": "SEALED",
    "experiment_count": 4,
    "experiments": [
        record.copy()
        for record in experiment_registry
    ],
}

# 6. Save without overwriting an existing file
with v03_path.open("x", encoding="utf-8") as file:
    json.dump(
        registry_v03,
        file,
        indent=2,
        allow_nan=False,
    )

# 7. Reload and verify the exported registry
with v03_path.open("r", encoding="utf-8") as file:
    restored = json.load(file)

assert restored == registry_v03
assert restored["experiment_count"] == 4
assert restored["holdout_status"] == "SEALED"

# 8. Calculate SHA-256 checksum
digest = hashlib.sha256()

with v03_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        digest.update(chunk)

# 9. Display verification results
print("MILESTONE 06 — REGISTRY V03 EXPORT")
print("=" * 60)

print("Filename:", v03_path.name)
print("Registered experiments:", restored["experiment_count"])

print("\nVALIDATION PR-AUC")

for record in restored["experiments"]:
    print(
        f"{record['experiment_id']}: "
        f"{record['validation_pr_auc']:.6f}"
    )

print("\nSHA-256:", digest.hexdigest())

print("\nRegistry v03 created: PASS")
print("JSON reload verification: PASS")
print("Four experiment records: PASS")
print("Previous registry records unchanged: PASS")
print("Registry v01 and v02: PRESERVED")
print("Month-7 test set: SEALED")
print("Step 06.33F: PASS")
